# 06 — Sliding daily rollout (DT=360 h)

Self-contained: loads **u,v** from Zarr, runs trained adv-NO, saves per-shift PNGs (no separate export needed).

**Frame spacing:** 15 days (`DT=360` h). **Row layout:** cond `−30,−15,t` (small) | GT (large) | Pred (large) | error (small).

**Section 1 (backward 0…15):** anchor slides back 1d/step; ends at cond `t−45,t−30,t−15`, GT/Pred at `t`.

**Section 2 (forward 0…15):** anchor slides forward 1d/step; GT stays +15d ahead of anchor.

**Section 3 (model-pred rollout):** same cond as forward (−30, −15, ref+shift), but the third frame uses **model prediction at ref+0…+15d** (from section 1 backward cache) instead of data. Run section 1 first.

**Active model:** `adv_no_uv_23_dt360_vgg_warm10_adv60` (trained on 2023 export). Zarr anchor in **2023** (`REF_ANCHOR`). `# Old:` 13–22 model + 2022 anchor (commented in setup).

Set `REF_ANCHOR` in setup. Output: `PNG/slide_rollout_{RUN_NAME}/backward|forward|forward_model_pred_rollout/` + RMSE curves (`backward_rmse.png`, `forward_vs_model_pred_rollout_rmse.png`).


In [2]:
import json
import os
import re
import sys
from pathlib import Path

import cv2
import imageio.v2 as imageio
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from gUnet import Unet2D

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.data.useful_transforms import RotatedFixedCrop

# --- model (trained on 2023 export, DT=360) ---
RUN_NAME = "adv_no_uv_23_dt360_vgg_warm10_adv60"
RUN_DIR = Path("Models/adv_no") / RUN_NAME
MODEL_PATH = RUN_DIR / "best_model.pth"
STATS_PATH = RUN_DIR / "norm_stats.pt"

# --- old model ---
# RUN_NAME = "adv_no_uv_13_22_dt360_vgg_warm10_adv60"  # trained 2013–2022
# RUN_DIR = Path("Models/adv_no") / RUN_NAME
# MODEL_PATH = RUN_DIR / "best_model.pth"
# STATS_PATH = RUN_DIR / "norm_stats.pt"

DT = 360
DT_STEP_DAYS = DT // 24
iD = 2
HOUR_OF_DAY = 12

# Zarr + fix window (same as export notebooks; aligns with data/sample_23_smallfixwin_uv_dt360)
ZARR_PATH = os.getenv("GUIANA_ZARR_PATH", "s3://oceangen-data/margem_equatorial_tensor.zarr")
WINDOW_TOP, WINDOW_LEFT = 330, 70
CROP_HEIGHT, CROP_WIDTH = 128, 128
WINDOW_ANGLE = -45.0

# Reference anchor t (last conditioning time at shift=0) — 2023 eval period
REF_ANCHOR = pd.Timestamp("2023-06-15 12:00:00")
# REF_ANCHOR = pd.Timestamp("2023-01-10 12:00:00")  # alternate 2023 anchor

# --- old anchor (13–22 model era) ---
# REF_ANCHOR = pd.Timestamp("2022-06-15 12:00:00")

PNG_ROOT = Path("PNG") / f"slide_rollout_{RUN_NAME}"
MAX_SHIFT_BACK = 15   # cond reaches t-45,t-30,t-15 | GT reaches t
MAX_SHIFT_FWD = 15    # anchor t -> t+15 | GT t+15 -> t+30
SECTION1_DIR = PNG_ROOT / "backward"
SECTION2_DIR = PNG_ROOT / "forward"
SECTION3_DIR = PNG_ROOT / "forward_model_pred_rollout"
BACKWARD_PRED_CACHE: dict[int, dict] = {}  # shift -> {pred_uv, pred_time, mask, anchor}
for d in (SECTION1_DIR, SECTION2_DIR, SECTION3_DIR):
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

stats = torch.load(STATS_PATH, map_location="cpu")
mean_y = stats["mean"]  # [1, iD, 1, 1]
std_y = stats["std"]
mean_x = mean_y.repeat(1, 3, 1, 1)
std_x = std_y.repeat(1, 3, 1, 1)

crop = RotatedFixedCrop(
    size=(CROP_HEIGHT, CROP_WIDTH),
    top=WINDOW_TOP,
    left=WINDOW_LEFT,
    angle_deg=WINDOW_ANGLE,
)
zarr_ds = GuianaDataloader(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=3,
    forecast_steps=1,
    transform=crop,
)

Par = {
    "lb": 3,
    "nf": iD,
    "nx": CROP_HEIGHT,
    "ny": CROP_WIDTH,
    "inp_shift": 0.0,
    "inp_scale": 1.0,
    "out_shift": 0.0,
    "out_scale": 1.0,
    "t_shift": 0.0,
    "t_scale": 1.0,
}
model = Unet2D(dim=32, Par=Par, dim_mults=(1, 2, 4, 8), channels=Par["lb"] * Par["nf"])
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))
model = model.to(device).eval()

print(f"Model: {MODEL_PATH}")
print(f"Reference anchor t: {REF_ANCHOR}")
print(f"Zarr times available: {len(zarr_ds.time_to_idx)}")


def _time_feature(ts: pd.Timestamp) -> torch.Tensor:
    """Match training + 05_evaluate_adv_no_rollout time embedding."""
    d = ts.dayofyear / 365.0
    h = ts.hour / 365.0
    return torch.tensor(d + h / 365.0, dtype=torch.float32)


def _load_cropped_uv_mask(ts: pd.Timestamp):
    """Return u,v [2,H,W], mask [H,W] (1=invalid)."""
    t = pd.Timestamp(ts)
    if t not in zarr_ds.time_to_idx:
        raise KeyError(f"Timestamp not in Zarr: {t}")
    fields = zarr_ds.load_uvssh_frame(t).astype(np.float32, copy=False)
    fields = np.nan_to_num(fields, nan=0.0)
    h, w = fields.shape[-2:]
    mask_ch = (~np.isfinite(fields[:1])).astype(np.float32)
    bat = np.ones((1, h, w), dtype=np.float32)
    lat, lon = zarr_ds._latlon_grid(h, w)
    bundle = torch.cat(
        [
            torch.tensor(mask_ch, dtype=torch.float32).unsqueeze(0),
            torch.tensor(fields, dtype=torch.float32).unsqueeze(0),
            torch.tensor(bat, dtype=torch.float32).unsqueeze(0),
            torch.tensor(lat, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
            torch.tensor(lon, dtype=torch.float32).unsqueeze(0).unsqueeze(0),
        ],
        dim=1,
    )
    bundle = crop(bundle)
    if not torch.is_tensor(bundle):
        bundle = torch.tensor(bundle, dtype=torch.float32)
    mask, data, _, _, _ = torch.split(bundle, [1, 3, 1, 1, 1], dim=1)
    uv = data[0, :2].clone()
    m = (mask[0, 0] > 0).numpy()
    return uv, m


def speed_mag(uv: np.ndarray) -> np.ndarray:
    return np.sqrt(uv[0] ** 2 + uv[1] ** 2)


def apply_mask(field: np.ndarray, mask: np.ndarray) -> np.ndarray:
    out = field.copy()
    out[mask] = np.nan
    return out


@torch.no_grad()
def predict_single_step(anchor: pd.Timestamp) -> tuple[np.ndarray, np.ndarray, list[pd.Timestamp]]:
    """Cond at anchor-30d,-15d,anchor; predict +15d state (normalized residual)."""
    times = [anchor - pd.Timedelta(days=30), anchor - pd.Timedelta(days=15), anchor]
    frames, masks = [], []
    for t in times:
        uv, m = _load_cropped_uv_mask(t)
        frames.append(uv)
        masks.append(m)
    mask = masks[-1]

    x = torch.cat(frames, dim=0).float().unsqueeze(0)
    x = (x - mean_x) / (std_x + 1e-8)
    x_dev = x.to(device)
    t_feat = _time_feature(anchor).view(1).to(device)
    pred = model(x_dev, t_feat)
    pred = pred + x_dev[:, -iD:]
    pred = pred * (std_y.to(device) + 1e-8) + mean_y.to(device)
    return pred[0].detach().cpu().numpy(), mask, times


@torch.no_grad()
def predict_from_cond_uv(frames_uv: list[np.ndarray], time_ts: pd.Timestamp) -> np.ndarray:
    """Run model on explicit 3-frame u,v conditioning; time embedding at time_ts."""
    x = torch.cat([torch.tensor(u, dtype=torch.float32) for u in frames_uv], dim=0).unsqueeze(0)
    x = (x - mean_x) / (std_x + 1e-8)
    x_dev = x.to(device)
    t_feat = _time_feature(time_ts).view(1).to(device)
    pred = model(x_dev, t_feat)
    pred = pred + x_dev[:, -iD:]
    pred = pred * (std_y.to(device) + 1e-8) + mean_y.to(device)
    return pred[0].detach().cpu().numpy()


def build_backward_pred_cache(max_shift: int = MAX_SHIFT_BACK) -> dict[int, dict]:
    """Re-run backward forecasts to populate BACKWARD_PRED_CACHE."""
    cache: dict[int, dict] = {}
    for shift in range(max_shift + 1):
        anchor = REF_ANCHOR - pd.Timedelta(days=shift)
        gt_time = REF_ANCHOR + pd.Timedelta(days=15 - shift)
        try:
            pred_uv, mask, _ = predict_single_step(anchor)
            cache[shift] = {"pred_uv": pred_uv, "pred_time": gt_time, "anchor": anchor, "mask": mask}
        except KeyError as e:
            print(f"cache skip shift={shift}: {e}")
    print(f"Built backward pred cache: {len(cache)} shifts")
    return cache


def ensure_backward_pred_cache() -> None:
    global BACKWARD_PRED_CACHE
    if len(BACKWARD_PRED_CACHE) >= MAX_SHIFT_FWD + 1:
        return
    BACKWARD_PRED_CACHE = build_backward_pred_cache()


def _prepare_sliding_row(
    cond_times: list[pd.Timestamp],
    gt_time: pd.Timestamp,
    pred_uv: np.ndarray,
    mask: np.ndarray,
    *,
    cond_uv_list: list[np.ndarray] | None = None,
    cond_labels: list[str] | None = None,
) -> dict:
    if cond_uv_list is None:
        cond_uv = []
        for t in cond_times:
            uv, m = _load_cropped_uv_mask(t)
            cond_uv.append(uv.numpy())
            mask = m
    else:
        cond_uv = cond_uv_list
    gt_uv = _load_cropped_uv_mask(gt_time)[0].numpy()

    cond_spd = [speed_mag(u) for u in cond_uv]
    gt_spd = speed_mag(gt_uv)
    pred_spd = speed_mag(pred_uv)
    err_spd = pred_spd - gt_spd

    def rel_label(ts: pd.Timestamp) -> str:
        return f"{(ts - REF_ANCHOR).days:+d}d"

    if cond_labels is None:
        cond_labels = [rel_label(cond_times[0]), rel_label(cond_times[1]), rel_label(cond_times[2])]
    panels = [
        (cond_spd[0], cond_labels[0]),
        (cond_spd[1], cond_labels[1]),
        (cond_spd[2], cond_labels[2]),
        (gt_spd, f"GT {rel_label(gt_time)}"),
        (pred_spd, "Pred"),
        (err_spd, "Pred−GT"),
    ]
    rmse = float(np.sqrt(np.nanmean(apply_mask(err_spd, mask) ** 2)))
    return {
        "panels": panels,
        "scale_fields": cond_spd + [gt_spd, pred_spd],
        "err_spd": err_spd,
        "rmse": rmse,
        "mask": mask,
    }


def _draw_sliding_row(
    fig,
    gs_row,
    row_data: dict,
    *,
    vmin: float,
    vmax: float,
    err_lim: float,
):
    width_ratios = [1, 1, 1, 2.2, 2.2, 1]
    gs = gridspec.GridSpecFromSubplotSpec(
        1, 6, subplot_spec=gs_row, width_ratios=width_ratios, wspace=0.25
    )
    mask = row_data["mask"]
    for i, (field, label) in enumerate(row_data["panels"]):
        ax = fig.add_subplot(gs[i])
        f = apply_mask(field, mask)
        if i == 5:
            im = ax.imshow(f, cmap="RdBu_r", vmin=-err_lim, vmax=err_lim)
        else:
            im = ax.imshow(f, cmap="viridis", vmin=vmin, vmax=vmax)
        big = i in (3, 4)
        ax.set_title(label, fontsize=10 if big else 9, fontweight="bold" if big else "normal")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)


def _draw_row_header(fig, gs_hdr, text: str):
    ax = fig.add_subplot(gs_hdr)
    ax.axis("off")
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        transform=ax.transAxes,
    )


def plot_sliding_row(
    cond_times: list[pd.Timestamp],
    gt_time: pd.Timestamp,
    pred_uv: np.ndarray,
    mask: np.ndarray,
    title: str,
    save_path: Path,
    *,
    cond_uv_list: list[np.ndarray] | None = None,
    cond_labels: list[str] | None = None,
):
    row = _prepare_sliding_row(
        cond_times, gt_time, pred_uv, mask, cond_uv_list=cond_uv_list, cond_labels=cond_labels
    )
    mask = row["mask"]
    spd_fields = row["scale_fields"]
    vmin = np.nanmin([np.nanmin(apply_mask(s, mask)) for s in spd_fields])
    vmax = np.nanmax([np.nanmax(apply_mask(s, mask)) for s in spd_fields])
    err_lim = np.nanmax(np.abs(apply_mask(row["err_spd"], mask)))

    fig = plt.figure(figsize=(15, 3.2))
    gs = gridspec.GridSpec(1, 1, figure=fig)
    _draw_sliding_row(fig, gs[0], row, vmin=vmin, vmax=vmax, err_lim=err_lim)
    fig.suptitle(
        f"{title}\nref t={REF_ANCHOR:%Y-%m-%d %Hh} | speed RMSE={row['rmse']:.4f}",
        fontsize=11,
        y=1.05,
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {save_path}")


def plot_sliding_dual_row(
    row1: dict,
    row2: dict,
    row_titles: tuple[str, str],
    suptitle: str,
    save_path: Path,
):
    mask = row1["mask"]
    spd_fields = row1["scale_fields"] + row2["scale_fields"]
    vmin = np.nanmin([np.nanmin(apply_mask(s, mask)) for s in spd_fields])
    vmax = np.nanmax([np.nanmax(apply_mask(s, mask)) for s in spd_fields])
    err_lim = max(
        float(np.nanmax(np.abs(apply_mask(row1["err_spd"], mask)))),
        float(np.nanmax(np.abs(apply_mask(row2["err_spd"], mask)))),
    )

    hdr1 = f"{row_titles[0]}  ·  RMSE {row1['rmse']:.4f}"
    hdr2 = f"{row_titles[1]}  ·  RMSE {row2['rmse']:.4f}"

    fig = plt.figure(figsize=(15, 6.6))
    gs = gridspec.GridSpec(
        4,
        1,
        figure=fig,
        height_ratios=[0.07, 1, 0.07, 1],
        hspace=0.28,
        top=0.94,
        bottom=0.04,
        left=0.02,
        right=0.98,
    )
    _draw_row_header(fig, gs[0], hdr1)
    _draw_sliding_row(fig, gs[1], row1, vmin=vmin, vmax=vmax, err_lim=err_lim)
    _draw_row_header(fig, gs[2], hdr2)
    _draw_sliding_row(fig, gs[3], row2, vmin=vmin, vmax=vmax, err_lim=err_lim)
    fig.suptitle(
        f"{suptitle}  ·  ref t={REF_ANCHOR:%Y-%m-%d %Hh}",
        fontsize=11,
        y=0.98,
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight", pad_inches=0.08)
    plt.close(fig)
    print(f"Saved {save_path}")


def run_backward_section(max_shift: int = MAX_SHIFT_BACK):
    """Anchor slides back 1d/step; GT slides from t+15d down to t."""
    for shift in range(max_shift + 1):
        anchor = REF_ANCHOR - pd.Timedelta(days=shift)
        gt_time = REF_ANCHOR + pd.Timedelta(days=15 - shift)
        cond_times = [anchor - pd.Timedelta(days=30), anchor - pd.Timedelta(days=15), anchor]
        try:
            pred_uv, mask, _ = predict_single_step(anchor)
            _load_cropped_uv_mask(gt_time)
            BACKWARD_PRED_CACHE[shift] = {
                "pred_uv": pred_uv,
                "pred_time": gt_time,
                "anchor": anchor,
                "mask": mask,
            }
        except KeyError as e:
            print(f"skip backward shift={shift}: {e}")
            continue
        fname = SECTION1_DIR / f"shift_{shift:03d}_anchor_{anchor:%Y%m%d}_gt_{gt_time:%Y%m%d}.png"
        plot_sliding_row(
            cond_times,
            gt_time,
            pred_uv,
            mask,
            f"Backward slide −{shift}d | anchor {anchor:%Y-%m-%d}",
            fname,
        )

    plot_backward_rmse_curves(max_shift)


def run_forward_section(max_shift: int = MAX_SHIFT_FWD):
    """Anchor slides forward 1d/step; GT stays +15d ahead of anchor."""
    for shift in range(max_shift + 1):
        anchor = REF_ANCHOR + pd.Timedelta(days=shift)
        gt_time = anchor + pd.Timedelta(days=15)
        cond_times = [anchor - pd.Timedelta(days=30), anchor - pd.Timedelta(days=15), anchor]
        try:
            pred_uv, mask, _ = predict_single_step(anchor)
            _load_cropped_uv_mask(gt_time)
        except KeyError as e:
            print(f"skip forward shift={shift}: {e}")
            continue
        fname = SECTION2_DIR / f"shift_{shift:03d}_anchor_{anchor:%Y%m%d}_gt_{gt_time:%Y%m%d}.png"
        plot_sliding_row(
            cond_times,
            gt_time,
            pred_uv,
            mask,
            f"Forward slide +{shift}d | anchor {anchor:%Y-%m-%d}",
            fname,
        )




def collect_backward_rmse(max_shift: int = MAX_SHIFT_BACK) -> tuple[list[int], list[float], list[float]]:
    """Compute model and persistence speed RMSE for backward slide at each shift."""
    shifts_done: list[int] = []
    rmse_vals: list[float] = []
    persist_rmse: list[float] = []
    for shift in range(max_shift + 1):
        anchor = REF_ANCHOR - pd.Timedelta(days=shift)
        gt_time = REF_ANCHOR + pd.Timedelta(days=15 - shift)
        try:
            pred_uv, mask, _ = predict_single_step(anchor)
            _load_cropped_uv_mask(gt_time)
        except KeyError:
            continue
        shifts_done.append(shift)
        rmse_vals.append(_speed_rmse(pred_uv, gt_time, mask))
        persist_rmse.append(_persistence_rmse(anchor, gt_time, mask))
    return shifts_done, rmse_vals, persist_rmse


def plot_backward_rmse(
    shifts: list[int],
    rmse_vals: list[float],
    persist_rmse: list[float],
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(shifts, rmse_vals, "o-", label="Backward slide", linewidth=2, markersize=6, color="C2")
    ax.plot(shifts, persist_rmse, "^--", label="Persistence (+15d)", linewidth=2, markersize=6, color="C3")
    ax.set_xlabel("Backward shift (days from ref anchor)")
    ax.set_ylabel("Speed RMSE")
    ax.set_title(f"Backward slide RMSE | ref t={REF_ANCHOR:%Y-%m-%d %Hh}")
    ax.set_xticks(shifts)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {save_path}")


def plot_backward_rmse_curves(max_shift: int = MAX_SHIFT_BACK) -> Path:
    """Save backward RMSE curve PNG without regenerating slide PNGs."""
    shifts, rmse_vals, persist_rmse = collect_backward_rmse(max_shift)
    out = PNG_ROOT / "backward_rmse.png"
    if not shifts:
        print("No backward shifts processed — RMSE curve not saved")
        return out
    plot_backward_rmse(shifts, rmse_vals, persist_rmse, out)
    return out


def _speed_rmse(pred_uv: np.ndarray, gt_time: pd.Timestamp, mask: np.ndarray) -> float:
    gt_uv = _load_cropped_uv_mask(gt_time)[0].numpy()
    err = speed_mag(pred_uv) - speed_mag(gt_uv)
    return float(np.sqrt(np.nanmean(apply_mask(err, mask) ** 2)))


def _persistence_rmse(persist_time: pd.Timestamp, gt_time: pd.Timestamp, mask: np.ndarray) -> float:
    """Persistence: hold u,v at persist_time constant to forecast gt_time (+15d baseline)."""
    persist_uv = _load_cropped_uv_mask(persist_time)[0].numpy()
    return _speed_rmse(persist_uv, gt_time, mask)


def plot_forward_vs_rollout_rmse(
    shifts: list[int],
    forward_rmse: list[float],
    rollout_rmse: list[float],
    persist_rmse: list[float],
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(shifts, forward_rmse, "o-", label="Forward (data cond)", linewidth=2, markersize=6)
    ax.plot(shifts, rollout_rmse, "s-", label="Model-pred rollout", linewidth=2, markersize=6)
    ax.plot(shifts, persist_rmse, "^--", label="Persistence (+15d)", linewidth=2, markersize=6, color="C3")
    ax.set_xlabel("Shift (days from ref anchor)")
    ax.set_ylabel("Speed RMSE")
    ax.set_title(f"Forward vs model-pred rollout RMSE | ref t={REF_ANCHOR:%Y-%m-%d %Hh}")
    ax.set_xticks(shifts)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {save_path}")




def collect_forward_vs_rollout_rmse(max_shift: int = MAX_SHIFT_FWD) -> tuple[list[int], list[float], list[float], list[float]]:
    """Compute forward, model-pred rollout, and persistence speed RMSE at each shift."""
    ensure_backward_pred_cache()
    shifts_done: list[int] = []
    forward_rmse: list[float] = []
    rollout_rmse: list[float] = []
    persist_rmse: list[float] = []

    for shift in range(max_shift + 1):
        back_shift = max_shift - shift
        if back_shift not in BACKWARD_PRED_CACHE:
            continue
        t = REF_ANCHOR + pd.Timedelta(days=shift)
        t_m30 = t - pd.Timedelta(days=30)
        t_m15 = t - pd.Timedelta(days=15)
        gt_time = t + pd.Timedelta(days=15)
        try:
            uv_m30, mask = _load_cropped_uv_mask(t_m30)
            uv_m15, _ = _load_cropped_uv_mask(t_m15)
            _load_cropped_uv_mask(t)
            _load_cropped_uv_mask(gt_time)
        except KeyError:
            continue
        model_pred = BACKWARD_PRED_CACHE[back_shift]["pred_uv"]
        fwd_pred, fwd_mask, _ = predict_single_step(t)
        rollout_pred = predict_from_cond_uv([uv_m30.numpy(), uv_m15.numpy(), model_pred], t)
        shifts_done.append(shift)
        forward_rmse.append(_speed_rmse(fwd_pred, gt_time, fwd_mask))
        rollout_rmse.append(_speed_rmse(rollout_pred, gt_time, mask))
        persist_rmse.append(_persistence_rmse(t, gt_time, mask))
    return shifts_done, forward_rmse, rollout_rmse, persist_rmse


def plot_forward_vs_rollout_rmse_curves(max_shift: int = MAX_SHIFT_FWD) -> Path:
    """Save RMSE curve PNG without regenerating section-3 slide PNGs."""
    shifts, fwd, roll, persist = collect_forward_vs_rollout_rmse(max_shift)
    out = PNG_ROOT / "forward_vs_model_pred_rollout_rmse.png"
    if not shifts:
        print("No shifts processed — RMSE curve not saved")
        return out
    plot_forward_vs_rollout_rmse(shifts, fwd, roll, persist, out)
    return out


def run_model_pred_rollout_section(max_shift: int = MAX_SHIFT_FWD):
    """Forward +15d rollout PNGs (2-row) + RMSE curve."""
    ensure_backward_pred_cache()
    for shift in range(max_shift + 1):
        back_shift = max_shift - shift
        if back_shift not in BACKWARD_PRED_CACHE:
            print(f"skip model-pred rollout shift={shift}: no model pred at ref+{shift}d (back_shift={back_shift})")
            continue
        t = REF_ANCHOR + pd.Timedelta(days=shift)
        t_m30 = t - pd.Timedelta(days=30)
        t_m15 = t - pd.Timedelta(days=15)
        gt_time = t + pd.Timedelta(days=15)
        cond_times = [t_m30, t_m15, t]
        try:
            uv_m30, mask = _load_cropped_uv_mask(t_m30)
            uv_m15, _ = _load_cropped_uv_mask(t_m15)
            _load_cropped_uv_mask(t)
            _load_cropped_uv_mask(gt_time)
        except KeyError as e:
            print(f"skip model-pred rollout shift={shift}: {e}")
            continue

        model_pred = BACKWARD_PRED_CACHE[back_shift]["pred_uv"]
        fwd_pred, fwd_mask, _ = predict_single_step(t)
        data_cond_labels = [
            f"{(t_m30 - REF_ANCHOR).days:+d}d",
            f"{(t_m15 - REF_ANCHOR).days:+d}d",
            f"+{shift}d Data",
        ]
        rollout_cond_uv = [uv_m30.numpy(), uv_m15.numpy(), model_pred]
        rollout_pred = predict_from_cond_uv(rollout_cond_uv, t)
        rollout_cond_labels = [
            f"{(t_m30 - REF_ANCHOR).days:+d}d",
            f"{(t_m15 - REF_ANCHOR).days:+d}d",
            f"+{shift}d Model Prediction",
        ]

        fname = SECTION3_DIR / f"shift_{shift:03d}_anchor_{t:%Y%m%d}_gt_{gt_time:%Y%m%d}.png"
        row1 = _prepare_sliding_row(cond_times, gt_time, fwd_pred, fwd_mask, cond_labels=data_cond_labels)
        row2 = _prepare_sliding_row(
            cond_times, gt_time, rollout_pred, mask,
            cond_uv_list=rollout_cond_uv, cond_labels=rollout_cond_labels,
        )
        plot_sliding_dual_row(
            row1, row2,
            row_titles=(
                f"Forward (data cond) +{shift}d | anchor {t:%Y-%m-%d}",
                f"Model-pred rollout +{shift}d | cond −30,−15,+{shift}(model) | t={t:%Y-%m-%d}",
            ),
            suptitle=f"Forward vs model-pred rollout | shift +{shift}d",
            save_path=fname,
        )

    plot_forward_vs_rollout_rmse_curves(max_shift)

SECONDS_PER_SLIDE = 2.0
VIDEO_FPS = 24
FRAMES_PER_SLIDE = int(SECONDS_PER_SLIDE * VIDEO_FPS)


def _shift_index(path: Path) -> int:
    m = re.search(r"shift_(\d+)", path.name)
    return int(m.group(1)) if m else 0


def make_slide_video(image_dir: Path, out_path: Path) -> None:
    paths = sorted(image_dir.glob("shift_*.png"), key=_shift_index)
    if not paths:
        raise FileNotFoundError(f"No shift_*.png in {image_dir}")

    first = imageio.imread(paths[0])
    h, w = first.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, VIDEO_FPS, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for {out_path}")

    for path in paths:
        rgb = imageio.imread(path)
        if rgb.ndim == 2:
            rgb = cv2.cvtColor(rgb, cv2.COLOR_GRAY2BGR)
        else:
            rgb = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        if rgb.shape[1] != w or rgb.shape[0] != h:
            rgb = cv2.resize(rgb, (w, h))
        for _ in range(FRAMES_PER_SLIDE):
            writer.write(rgb)

    writer.release()
    duration = len(paths) * SECONDS_PER_SLIDE
    print(f"Saved {out_path}")
    print(f"  {len(paths)} slides | {SECONDS_PER_SLIDE}s each | {duration:.0f}s total | {VIDEO_FPS} fps")


print("Setup OK")


/tmp/ipykernel_3575792/2574527981.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  stats = torch.load(STATS_PATH, map_location="cpu")


Model: Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/best_model.pth
Reference anchor t: 2023-06-15 12:00:00
Zarr times available: 96385
Setup OK


## Section 1 — backward daily slide (0…15)

16 PNGs: reference `t` fixed; anchor and GT move back 1 day per step.


In [3]:
run_backward_section(max_shift=MAX_SHIFT_BACK)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_000_anchor_20230615_gt_20230630.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_001_anchor_20230614_gt_20230629.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_002_anchor_20230613_gt_20230628.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_003_anchor_20230612_gt_20230627.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_004_anchor_20230611_gt_20230626.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_005_anchor_20230610_gt_20230625.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_006_anchor_20230609_gt_20230624.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_007_anchor_20230608_gt_20230623.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward/shift_008_anchor_20230607_gt_20230622.png
S

## RMSE curve — backward slide

Saved to `PNG/slide_rollout_{RUN_NAME}/backward_rmse.png` (model + **persistence +15d** baseline).

Auto-generated at the end of section 1, or run the cell below on its own.


In [4]:
plot_backward_rmse_curves(max_shift=MAX_SHIFT_BACK)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward_rmse.png


PosixPath('PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward_rmse.png')

## Section 2 — forward daily slide (0…15)

Same reference `REF_ANCHOR`; anchor moves forward 1 day per plot.


In [5]:
run_forward_section(max_shift=MAX_SHIFT_FWD)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_000_anchor_20230615_gt_20230630.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_001_anchor_20230616_gt_20230701.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_002_anchor_20230617_gt_20230702.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_003_anchor_20230618_gt_20230703.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_004_anchor_20230619_gt_20230704.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_005_anchor_20230620_gt_20230705.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_006_anchor_20230621_gt_20230706.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_007_anchor_20230622_gt_20230707.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward/shift_008_anchor_20230623_gt_20230708.png
Saved PNG/

## Section 3 — model-prediction rollout (+15d forecast)

Same sliding window as section 2: at shift `k`, anchor `t = ref + k`, cond frames at **−30d, −15d** from data.

The **third frame** (normally data at `ref + k`, i.e. +0…+15d) is replaced by the **model prediction** at that same offset — backward shift `15−k` predicts `ref + k`.

Each PNG has **2 rows**: row 1 = standard forward (all data); row 2 = model-pred rollout (third frame = model prediction).

Run section 1 first (fills cache), or setup rebuilds it automatically.

In [6]:
run_model_pred_rollout_section(max_shift=MAX_SHIFT_FWD)

Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_000_anchor_20230615_gt_20230630.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_001_anchor_20230616_gt_20230701.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_002_anchor_20230617_gt_20230702.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_003_anchor_20230618_gt_20230703.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_004_anchor_20230619_gt_20230704.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_005_anchor_20230620_gt_20230705.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_006_anchor_20230621_gt_20230706.png
Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout/shift_007_

## RMSE curves — Forward vs model-pred rollout

Saved to `PNG/slide_rollout_{RUN_NAME}/forward_vs_model_pred_rollout_rmse.png` (forward, model-pred rollout, and **persistence +15d** per shift).

Auto-generated at the end of section 3, or run the cell below on its own.


In [7]:
plot_forward_vs_rollout_rmse_curves(max_shift=MAX_SHIFT_FWD)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_vs_model_pred_rollout_rmse.png


PosixPath('PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_vs_model_pred_rollout_rmse.png')

In [8]:
model_pred_video = PNG_ROOT / "forward_model_pred_rollout_slides.mp4"
make_slide_video(SECTION3_DIR, model_pred_video)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_model_pred_rollout_slides.mp4
  16 slides | 2.0s each | 32s total | 24 fps


## Videos from slide PNGs

Run after sections 1 / 2 / 3. Each slide is shown for **2 seconds** (48 frames at 24 fps).


In [9]:
backward_video = PNG_ROOT / "backward_slides.mp4"
make_slide_video(SECTION1_DIR, backward_video)


Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/backward_slides.mp4
  16 slides | 2.0s each | 32s total | 24 fps


In [10]:
forward_video = PNG_ROOT / "forward_slides.mp4"
make_slide_video(SECTION2_DIR, forward_video)

Saved PNG/slide_rollout_adv_no_uv_23_dt360_vgg_warm10_adv60/forward_slides.mp4
  16 slides | 2.0s each | 32s total | 24 fps
